In [6]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.

In [7]:
from langchain_core.documents import Document

In [8]:
sample_doc = Document(
    page_content="Hello World!",
    metadata={"source": "https://www.google.com"}
)

In [9]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [10]:
type(sample_doc)

langchain_core.documents.base.Document

In [11]:
# Text data
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("python.txt", encoding="utf-8")

/tmp/ipykernel_3311/424300118.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [12]:
document = loader.load()

In [13]:
document

[Document(metadata={'source': 'python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and more.\n

## Ingestion Pipeline

In [14]:
# Data => Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [15]:
# Documents

In [16]:
def load_all_pdfs():
    folder_path = "pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [17]:
import os
import shutil

# Create the 'pdfs' directory if it doesn't exist
if not os.path.exists('pdfs'):
    os.makedirs('pdfs')
    print("Created directory: pdfs")

# List of PDF files to move from /content/ to the 'pdfs' directory
pdf_files_to_move = [
    "research2 (1).pdf",
    "Artificial Intelligence, Machine Learning, and Deep Learning.pdf"
]

# Move the PDF files
for pdf_file in pdf_files_to_move:
    source_path = os.path.join("/content/", pdf_file)
    destination_path = os.path.join("pdfs", pdf_file)
    if os.path.exists(source_path):
        # Check if the file is already in the destination to avoid error
        if not os.path.exists(destination_path):
            shutil.move(source_path, destination_path)
            print(f"Moved {pdf_file} to pdfs/")
        else:
            print(f"File {pdf_file} already exists in pdfs/")
    else:
        print(f"Warning: {pdf_file} not found at {source_path}")

all_pdf_documents = load_all_pdfs()

Created directory: pdfs
Moved research2 (1).pdf to pdfs/
Moved Artificial Intelligence, Machine Learning, and Deep Learning.pdf to pdfs/
total pdfs: 2
total pages: 360


In [18]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

In [19]:
# chunks
!pip install langchain_text_splitters

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [27]:
chunks = split_docs(all_pdf_documents)

In [28]:
len(chunks)

1502

In [29]:
## Embedding

In [30]:
from sentence_transformers import SentenceTransformer

In [31]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):

        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [32]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding dimensions= 384


/tmp/ipykernel_3311/4021223045.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


In [33]:
## Vector Store

import chromadb
import uuid

In [34]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [35]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [36]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

embeddings shape: (1502, 384)
total documents added in vector store= 1502
docs in collection: 1502


# Retrieval Pipeline

In [37]:
from sklearn.metrics.pairwise import cosine_similarity

In [38]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [39]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [40]:
rag_retriever.retrieve("Explain AI ML DL fundamental?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_a5a988c1-4813-47fc-bae8-8ac0d80bff28',
  'document': 'major parts of AI, which include machine learning and deep learning.\nMajor Parts of AI \nThe subsequent chapters in this book delve into various important parts of \nAI, which include:\n•\t ML (Machine Learning)\n•\t DL (Deep Learning)\n•\t NLP (Natural Language Processing)\n•\t RL (Reinforcement Learning)\n•\t DRL (Deep Reinforcement Learning)\nTraditional AI (twentieth century) is based on collections of rules, which \nled to expert systems in the 1980s. Traditional AI also involved LISP , which',
  'metadata': {'moddate': '2021-06-12T15:25:17+03:00',
   'total_pages': 339,
   'doc_index': 391,
   'creator': 'Adobe InDesign CS6 (Windows)',
   'page_label': '18',
   'creationdate': '2020-01-22T14:34:12+05:30',
   'page': 36,
   'producer': 'Adobe PDF Library 10.0.1',
   'trapped': '/False',
   'source': 'pdfs/Artificial Intelligence, Machine Learning, and Deep Learning.pdf',
   'content_length': 488},
  'distance': 0.

In [41]:
rag_retriever.retrieve("who is mayur zolekar?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 0 documents


[]